# 03 · Modelado: construcción y comparación del extractor
**Proyecto integrador — Lector de Facturas** · Sebastián Rojas y Jesús López

**Aclaración importante:** en este proyecto no se entrenan pesos. El OCR es Tesseract preentrenado (redes LSTM `tessdata_fast` y `tessdata_best`, fijadas por SHA-256) y la asignación de campos se hace con reglas, geometría y consenso entre lecturas. Por eso aquí el "modelado" consiste en **construir el extractor y comparar versiones** con métricas sobre documentos de validación, todas con **las mismas lecturas OCR**:

| Modelo | Qué es |
|---|---|
| **M0 · Reglas iniciales** | Solo `candidate_rules.extract` (reglas V5) sobre una lectura: la vista principal. |
| **M1 · V7 anterior** | Consenso de las 5 vistas `fast` del modelo anterior, con todas las etapas nuevas apagadas. |
| **M2 · Actual sin resumen contable** | Las 10 vistas y todas las etapas, salvo la contabilidad observada y el rescate por confianza. |
| **M3 · Resumen contable** | Modelo final (`configs/definitivo.json`). |

Después se hace una **ablación**: se apaga una etapa, un grupo de vistas o un mecanismo del consenso por vez para ver cuánto aporta. La evaluación final está en `05_evaluacion.ipynb`.

**Contenido:** 1. Configuración · 2. Muestra de validación · 3. OCR · 4. Modelos · 5. Comparación · 6. Ablación · 7. Análisis de errores · 8. Tiempo · 9. Modelo final

## 1. Configuración

In [ ]:
# ===== Configuración común de los notebooks 01–06 =====
# En Colab no hace falta clonar nada: se descarga el código de GitHub y se registra su commit.
USAR_CODIGO_LOCAL = False  # True si abres Jupyter dentro de un clon local del repositorio
CODIGO_LOCAL = ''          # Opcional: ruta del clon local (por defecto, la carpeta padre de notebooks/)
USAR_DRIVE = False         # True: datos, pesos, caché OCR y resultados quedan en Google Drive (los notebooks no repiten descargas ni OCR)
DESCARGAR_SI_FALTAN = True
DATOS_MANUAL = ''          # Opcional: ruta a data/raw existente en Drive o Colab
REPOSITORIO = 'Jesusl2202/Document-Scanner'
RAMA = 'main'
SEED = 42

from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib, zipfile, io, urllib.request


def comando(args, cwd=None, mostrar_error=True):
    """Ejecuta un comando y muestra la salida solo si falla."""
    r = subprocess.run(list(map(str, args)), cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        if mostrar_error:
            print(r.stdout[-3000:]); print(r.stderr[-3000:])
        r.check_returncode()
    return r.stdout + r.stderr


# 1) Carpeta de trabajo (fuera de Git)
if USAR_CODIGO_LOCAL:
    CODIGO = Path(CODIGO_LOCAL) if CODIGO_LOCAL else (Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
    TRABAJO = CODIGO / 'outputs/experimentos'
elif USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    TRABAJO = Path('/content/drive/MyDrive/Lector_Facturas_Experimentos')
else:
    TRABAJO = Path('/content/Lector_Facturas_Experimentos')
TRABAJO.mkdir(parents=True, exist_ok=True)
SALIDAS = TRABAJO / 'notebooks'
SALIDAS.mkdir(parents=True, exist_ok=True)

# 2) Código del repositorio: se descarga el commit exacto y se registra
if USAR_CODIGO_LOCAL:
    REVISION = {'modo': 'local', 'ruta': str(CODIGO)}
else:
    pedido = urllib.request.Request(f'https://api.github.com/repos/{REPOSITORIO}/commits/{RAMA}', headers={'User-Agent': 'lector-facturas'})
    with urllib.request.urlopen(pedido, timeout=60) as respuesta:
        commit = json.load(respuesta)['sha']
    CODIGO = TRABAJO / f'codigo_{commit[:7]}'
    if not (CODIGO / 'src/lector_facturas/consensus.py').exists():
        with urllib.request.urlopen(f'https://codeload.github.com/{REPOSITORIO}/zip/{commit}', timeout=120) as respuesta:
            blob = respuesta.read()
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            for nombre in z.namelist():
                relativo = '/'.join(nombre.split('/')[1:])
                if not relativo or nombre.endswith('/'):
                    continue
                destino = (CODIGO / relativo).resolve()
                if not destino.is_relative_to(CODIGO.resolve()):
                    raise ValueError('Ruta no permitida dentro del ZIP')
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(z.read(nombre))
    REVISION = {'repositorio': REPOSITORIO, 'rama': RAMA, 'commit': commit}
assert (CODIGO / 'src/lector_facturas/consensus.py').exists(), 'No se encontró el código del lector.'
RAIZ = CODIGO
print('Código:', REVISION)

# 3) Tesseract y bibliotecas
if shutil.which('apt-get') and not shutil.which('tesseract'):
    print('Instalando Tesseract...')
    comando(['apt-get', 'update', '-qq'])
    comando(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr'])
assert shutil.which('tesseract'), 'No se encontró Tesseract. Usa Google Colab con CPU o instálalo en tu equipo.'
comando([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODIGO / 'requirements.txt'])
sys.path.insert(0, str(CODIGO / 'src'))
for modulo in list(sys.modules):
    if modulo == 'lector_facturas' or modulo.startswith('lector_facturas.'):
        del sys.modules[modulo]

# 4) Modelo: configuración definitiva + pesos Tesseract fijados por SHA-256 (fast, best y español)
from lector_facturas import frozen_runtime
RAW_CONFIG = json.loads((CODIGO / 'configs/definitivo.json').read_text(encoding='utf-8'))
assert hasattr(frozen_runtime, 'bind_models') and RAW_CONFIG.get('observed_accounting'), (
    'El código descargado no corresponde al modelo «resumen contable» (configs/definitivo.json). '
    'Revisa REPOSITORIO y RAMA, o usa USAR_CODIGO_LOCAL=True dentro de un clon del repositorio.')
RAW_CONFIG['seed'] = SEED
CONFIG = frozen_runtime.bind_models(RAW_CONFIG, TRABAJO / 'modelos')
VISTAS_MODELO = len(CONFIG['ocr']['ensemble_views'])
print(comando(['tesseract', '--version']).splitlines()[0], f'| {VISTAS_MODELO} vistas OCR + hasta',
      CONFIG['ocr']['adaptive_rescue']['max_views'], 'de rescate')

# 5) Datos: se buscan por contenido y se descargan si faltan
candidatos = [Path(DATOS_MANUAL)] if DATOS_MANUAL else [
    Path('/content/Document-Scanner/data/raw'), TRABAJO / 'data/raw']
DATOS = next((p for p in candidatos if p.exists()), candidatos[-1])
DATOS.mkdir(parents=True, exist_ok=True)


def localizar():
    """Encuentra la raíz de SROIE y de WildReceipt por su contenido, no por el nombre de la carpeta."""
    sroie = next((p.parent.parent for p in DATOS.rglob('entities') if p.is_dir() and p.parent.name == 'train'
                  and (p.parent / 'img').exists() and (p.parent / 'box').exists()), None)
    wild = next((p.parent for p in DATOS.rglob('class_list.txt') if (p.parent / 'train.txt').exists() and (p.parent / 'test.txt').exists()), None)
    return sroie, wild


SROIE, WILD = localizar()
for nombre, ruta in [('sroie', SROIE), ('wildreceipt', WILD)]:
    if ruta is None and DESCARGAR_SI_FALTAN:
        print('Descargando', nombre, '...', flush=True)
        try:
            print(comando([sys.executable, CODIGO / 'scripts/download_data.py', nombre, '--out', DATOS], mostrar_error=False))
        except subprocess.CalledProcessError:
            print(f'No se pudo descargar {nombre}. Descárgalo a mano en {DATOS} o indica DATOS_MANUAL (ver data/raw/README.md).')
SROIE, WILD = localizar()
print('SROIE:      ', SROIE or 'NO ENCONTRADO')
print('WildReceipt:', WILD or 'NO ENCONTRADO')
assert SROIE and WILD, 'Faltan datasets: revisa data/raw/README.md.'

# 6) Manifiesto con la partición del proyecto (train / validation / test / quarantine)
from lector_facturas.datasets import prepare
FILAS = prepare(SROIE, WILD, SALIDAS / 'preparados', RAW_CONFIG)
REVISADOS = json.loads((CODIGO / 'configs/documentos_revisados.json').read_text(encoding='utf-8'))
print('Documentos en el manifiesto:', len(FILAS), '| documentos ya revisados en el desarrollo:', len(REVISADOS))

In [ ]:
# Funciones de gráficos compartidas por los notebooks 01–06 (se repiten para que cada uno funcione solo en Colab).
import glob, re, json, types
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

AZUL, NARANJA, AQUA, GRIS = '#2a78d6', '#eb6834', '#1baf7a', '#8a8984'

FONDO, TEXTO, TEXTO2, REJILLA = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df'

COLOR = {'sroie': AZUL, 'wildreceipt': NARANJA}

NOMBRE = {'sroie': 'SROIE', 'wildreceipt': 'WildReceipt'}

CAMPOS = ['tax_id', 'date', 'document_number', 'subtotal', 'tax', 'total']

plt.rcParams.update({
    'figure.facecolor': FONDO, 'axes.facecolor': FONDO, 'axes.edgecolor': REJILLA,
    'axes.labelcolor': TEXTO2, 'xtick.color': TEXTO2, 'ytick.color': TEXTO2, 'text.color': TEXTO,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': REJILLA, 'grid.linewidth': .6, 'axes.axisbelow': True, 'legend.frameon': False})

def es_num(valor, decimales=0):
    """Formato numérico en español: 1.234,5."""
    texto = f'{valor:,.{decimales}f}'
    return texto.replace(',', 'X').replace('.', ',').replace('X', '.')

def guardar(fig, carpeta, nombre):
    fig.tight_layout()
    fig.savefig(Path(carpeta) / nombre, dpi=150)
    plt.close(fig)
    print('  figura:', nombre)

def leer_csv(ruta):
    return pd.read_csv(ruta, encoding='utf-8-sig')

def fig_particiones(documentos, incidencias, manifest, salida):
    """Figura 1. Documentos por partición y dataset."""
    if manifest is not None:
        conteo = manifest.groupby(['dataset', 'split']).size().unstack(fill_value=0)
    else:
        # Sin manifiesto solo se conocen desarrollo y cuarentena.
        conteo = documentos.groupby(['dataset', 'split']).size().unstack(fill_value=0)
        cuarentena = incidencias[incidencias.split == 'quarantine'].drop_duplicates('id')
        conteo['quarantine'] = cuarentena.groupby('dataset').size().reindex(conteo.index, fill_value=0)
        print('  aviso: sin --manifest la Figura 1 no muestra el test oficial')
    columnas = {'train': ('Entrenamiento', AZUL), 'validation': ('Validación', AQUA),
                'test': ('Test oficial (reservado)', GRIS), 'quarantine': ('Cuarentena', NARANJA)}
    total = int(conteo.values.sum())
    fig, ax = plt.subplots(figsize=(8, 3.2))
    izquierda = np.zeros(len(conteo))
    etiquetas = [NOMBRE.get(d, d) for d in conteo.index]
    for clave, (texto, color) in columnas.items():
        if clave not in conteo:
            continue
        valores = conteo[clave].values
        ax.barh(etiquetas, valores, left=izquierda, color=color, label=texto,
                edgecolor=FONDO, linewidth=2, height=.55)
        for i, v in enumerate(valores):
            if v > 60:
                ax.text(izquierda[i] + v / 2, i, es_num(v), ha='center', va='center',
                        color='white', fontsize=9, fontweight='bold')
        izquierda += valores
    ax.set_xlabel('Documentos')
    ax.set_title(f'Figura 1. Partición de los {es_num(total)} documentos', loc='left')
    ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(.5, -.28))
    ax.grid(axis='y', visible=False)
    ax.invert_yaxis()
    guardar(fig, salida, '01_particiones.png')
    return conteo

def fig_resolucion(documentos, salida):
    """Figura 2. Ancho vs. alto de las imágenes de desarrollo."""
    fig, ax = plt.subplots(figsize=(7, 4.6))
    for ds in ['wildreceipt', 'sroie']:
        g = documentos[documentos.dataset == ds]
        if g.empty:
            continue
        ax.scatter(g.width, g.height, s=12, alpha=.45, color=COLOR[ds], edgecolors='none',
                   label=f'{NOMBRE[ds]} (mediana {int(g.width.median())}×{int(g.height.median())} px)')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Ancho (px, escala log)')
    ax.set_ylabel('Alto (px, escala log)')
    ax.set_title('Figura 2. Resolución de las imágenes de desarrollo', loc='left')
    ax.legend(loc='upper left')
    guardar(fig, salida, '02_resolucion.png')

def fig_regiones(documentos, salida):
    """Figura 3. Regiones de texto anotadas por documento."""
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    bins = np.arange(0, documentos.regions.max() + 6, 6)
    for ds in ['sroie', 'wildreceipt']:
        g = documentos[documentos.dataset == ds].regions
        if g.empty:
            continue
        ax.hist(g, bins=bins, alpha=.6, color=COLOR[ds], edgecolor=FONDO, linewidth=1,
                label=f'{NOMBRE[ds]} (mediana {int(g.median())})')
        ax.axvline(g.median(), color=COLOR[ds], ls='--', lw=1.5)
    ax.set_xlabel('Regiones de texto anotadas por documento')
    ax.set_ylabel('Documentos')
    ax.set_title('Figura 3. Cantidad de texto por recibo', loc='left')
    ax.legend()
    guardar(fig, salida, '03_regiones.png')

def fig_cobertura(cobertura, salida):
    """Figura 4. Porcentaje de documentos con etiqueta utilizable por campo."""
    tabla = cobertura.pivot(index='dataset', columns='field', values='coverage').reindex(columns=CAMPOS)
    mapa = LinearSegmentedColormap.from_list('azules', ['#f1f0ec', '#cfe0f6', '#7fb0ea', AZUL, '#0d3f80'])
    fig, ax = plt.subplots(figsize=(8, 1.3 + .65 * len(tabla)))
    ax.imshow(tabla.values, cmap=mapa, vmin=0, vmax=1, aspect='auto')
    ax.grid(False)
    ax.set_xticks(range(len(CAMPOS)), CAMPOS)
    ax.set_yticks(range(len(tabla)), [NOMBRE.get(d, d) for d in tabla.index])
    for i in range(tabla.shape[0]):
        for j in range(tabla.shape[1]):
            v = tabla.values[i, j]
            ax.text(j, i, f'{es_num(100 * v)}%', ha='center', va='center',
                    color='white' if v > .55 else TEXTO, fontweight='bold')
    for borde in ax.spines.values():
        borde.set_visible(False)
    ax.set_title('Figura 4. Campos con etiqueta utilizable (documentos de desarrollo)', loc='left')
    guardar(fig, salida, '04_cobertura.png')

def fig_clases_lineas(texto, salida):
    """Figura 5. Desbalance de clases por línea (datos del clasificador de Semana 3)."""
    conteo = pd.Series({k.strip(): int(v) for k, v in (p.split('=') for p in texto.split(','))}).sort_values()
    total = conteo.sum()
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.barh(conteo.index, conteo.values, height=.55,
            color=[GRIS if k.lower().startswith('otro') else AZUL for k in conteo.index])
    for i, v in enumerate(conteo.values):
        ax.text(v * 1.08, i, f'{es_num(v)} ({es_num(100 * v / total, 1)}%)', va='center', color=TEXTO2, fontsize=9)
    ax.set_xscale('log')
    ax.set_xlim(conteo.min() / 2, conteo.max() * 4)
    ax.set_xlabel('Líneas OCR en entrenamiento (escala log)')
    ax.grid(axis='y', visible=False)
    ax.set_title('Figura 5. Desbalance de clases por línea', loc='left')
    guardar(fig, salida, '05_desbalance_lineas.png')

def fig_cer(cer, documentos, salida):
    """Figura 6. CER del OCR frente al tamaño de la imagen."""
    datos = cer.merge(documentos, on=['id', 'dataset'])
    if datos.empty:
        print('  aviso: los documentos de cer.csv no están en documents.csv; se omite la Figura 6')
        return None
    datos['mpx'] = datos.width * datos.height / 1e6
    rho = datos[['cer', 'mpx']].corr(method='spearman').iloc[0, 1]
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for ds in ['sroie', 'wildreceipt']:
        g = datos[datos.dataset == ds]
        if g.empty:
            continue
        ax.scatter(g.mpx, g.cer, s=46, color=COLOR[ds], edgecolors=FONDO, linewidths=1.5,
                   label=f'{NOMBRE[ds]} (CER mediano {es_num(g.cer.median(), 2)})')
    ax.set_xscale('log')
    ax.set_xlabel('Tamaño de la imagen (megapíxeles, escala log)')
    ax.set_ylabel('CER del OCR')
    ax.set_title(f'Figura 6. Imágenes más pequeñas se leen peor (Spearman ρ = {es_num(rho, 2)})', loc='left')
    ax.legend()
    guardar(fig, salida, '06_cer_vs_resolucion.png')
    return datos

def tipo_error(errores):
    """Clasifica cada campo evaluado en correcto, omitido o valor incorrecto."""
    vacio = errores.predicted.isna() | (errores.predicted.astype(str).str.strip() == '')
    return np.where(errores.correct == 1, 'Correcto',
                    np.where(vacio, 'Omitido (no extrajo nada)', 'Valor incorrecto'))

def fig_errores(errores, salida):
    """Figura 7. Resultado por campo: correcto, valor incorrecto u omitido."""
    errores = errores.assign(tipo=tipo_error(errores))
    orden = ['Correcto', 'Valor incorrecto', 'Omitido (no extrajo nada)']
    tabla = errores.groupby(['dataset', 'field', 'tipo']).size().unstack(fill_value=0).reindex(columns=orden, fill_value=0)
    tabla.index = [f'{NOMBRE.get(d, d)} · {c}' for d, c in tabla.index]
    documentos = errores.id.nunique()
    fig, ax = plt.subplots(figsize=(8, 1.4 + .4 * len(tabla)))
    izquierda = np.zeros(len(tabla))
    for col, color in zip(orden, [AQUA, NARANJA, '#b9b8b2']):
        valores = tabla[col].values
        ax.barh(tabla.index, valores, left=izquierda, color=color, label=col, edgecolor=FONDO, linewidth=2, height=.6)
        for i, v in enumerate(valores):
            if v >= 2:
                ax.text(izquierda[i] + v / 2, i, str(v), ha='center', va='center', fontsize=9,
                        color=TEXTO if col.startswith('Omitido') else 'white')
        izquierda += valores
    ax.invert_yaxis()
    ax.set_xlabel('Campos anotados')
    ax.grid(axis='y', visible=False)
    ax.set_title(f'Figura 7. Resultado por campo ({documentos} documentos)', loc='left')
    ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(.5, -.2))
    guardar(fig, salida, '07_errores_linea_base.png')
    return errores

def fig_correlaciones(documentos, salida):
    """Figura 8. Correlación de Spearman entre variables de la imagen, por dataset."""
    variables = {'width': 'Ancho', 'height': 'Alto', 'mpx': 'Megapíxeles', 'regions': 'Regiones', 'aspect': 'Alto/ancho'}
    mapa = LinearSegmentedColormap.from_list('div', [NARANJA, '#f6c3ad', '#f1f0ec', '#a9c8f0', AZUL])
    bases = [d for d in ['sroie', 'wildreceipt'] if d in set(documentos.dataset)]
    fig, ejes = plt.subplots(1, len(bases), figsize=(4.5 * len(bases), 3.8), squeeze=False)
    matrices = {}
    for ax, ds in zip(ejes[0], bases):
        m = documentos[documentos.dataset == ds][list(variables)].corr(method='spearman')
        matrices[ds] = m
        ax.imshow(m.values, cmap=mapa, vmin=-1, vmax=1)
        ax.grid(False)
        ax.set_xticks(range(len(variables)), variables.values(), rotation=35, ha='right')
        ax.set_yticks(range(len(variables)), variables.values())
        for i in range(len(variables)):
            for j in range(len(variables)):
                v = m.values[i, j]
                ax.text(j, i, es_num(v, 2), ha='center', va='center', fontsize=8,
                        color='white' if abs(v) > .6 else TEXTO)
        ax.set_title(NOMBRE[ds])
        for borde in ax.spines.values():
            borde.set_visible(False)
    fig.suptitle('Figura 8. Correlación de Spearman entre variables de la imagen', x=.02, ha='left', fontweight='bold')
    guardar(fig, salida, '08_correlaciones.png')
    return matrices

PATRONES_FECHA = {'99/99/9999': 'dd/mm/aaaa', '99-99-99': 'dd-mm-aa', '99-99-9999': 'dd-mm-aaaa',
                  '99/99/99': 'dd/mm/aa', '99 A 9999': 'dd MES aaaa'}

def patron(fecha):
    """Convierte '25/12/2018' en '99/99/9999' y 'MAR' en 'A' para agrupar formatos."""
    return re.sub(r'\d', '9', re.sub(r'[A-Za-z]+', 'A', fecha))

def fig_formatos_fecha(carpeta, salida):
    """Figura 9. Formatos de fecha en las etiquetas de entrenamiento de SROIE."""
    fechas = []
    for archivo in sorted(glob.glob(str(Path(carpeta) / '*'))):
        if not archivo.endswith(('.json', '.txt')):
            continue
        try:
            fechas.append(str(json.loads(Path(archivo).read_text(encoding='utf-8-sig')).get('date', '')).strip())
        except (json.JSONDecodeError, UnicodeDecodeError):
            print('  aviso: no se pudo leer', archivo)
    if not fechas:
        print('  aviso: no hay etiquetas en --sroie-keys; se omite la Figura 9')
        return None
    formatos = Counter(patron(f) for f in fechas)
    ambiguas = 0
    for f in fechas:
        m = re.fullmatch(r'\(?(\d{1,2})[/.\-](\d{1,2})[/.\-](\d{2,4})\)?', f)
        if m and int(m.group(1)) <= 12 and int(m.group(2)) <= 12 and m.group(1) != m.group(2):
            ambiguas += 1
    agrupado = Counter()
    for p, n in formatos.items():
        agrupado[PATRONES_FECHA.get(p, 'otros')] += n
    otros = len([p for p in formatos if p not in PATRONES_FECHA])
    orden = [v for v in PATRONES_FECHA.values() if agrupado[v]] + (['otros'] if agrupado['otros'] else [])
    orden = orden[::-1]
    etiquetas = [f'Otros {otros} formatos' if o == 'otros' else o for o in orden]
    valores = [agrupado[o] for o in orden]
    n = len(fechas)
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    ax.barh(etiquetas, valores, color=AZUL, height=.55)
    ax.grid(axis='y', visible=False)
    for i, v in enumerate(valores):
        ax.text(v + n * .006, i, f'{v} ({es_num(100 * v / n, 1)}%)', va='center', color=TEXTO2, fontsize=9)
    ax.set_xlim(0, max(valores) * 1.2)
    ax.set_xlabel(f'Recibos de entrenamiento de SROIE (n = {n})')
    ax.set_title('Figura 9. Formatos de fecha en las etiquetas de SROIE', loc='left')
    guardar(fig, salida, '09_formatos_fecha.png')
    return {'recibos': n, 'formatos_distintos': len(formatos), 'fechas_dia_mes_ambiguas': ambiguas,
            'porcentaje_ambiguas': round(100 * ambiguas / n, 1),
            'formatos': dict(formatos.most_common())}

def tablas(documentos, salida):
    """Estadísticas descriptivas y outliers (criterio 1,5 × IQR) por dataset."""
    variables = ['width', 'height', 'aspect', 'mpx', 'regions']
    describe = documentos.groupby('dataset')[variables].describe(percentiles=[.25, .5, .75]).T.round(2)
    describe.to_csv(Path(salida) / 'estadisticas_descriptivas.csv', encoding='utf-8-sig')
    filas = []
    for ds, g in documentos.groupby('dataset'):
        for v in ['width', 'height', 'mpx', 'regions']:
            q1, q3 = g[v].quantile([.25, .75])
            limite_bajo, limite_alto = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
            n = int(((g[v] < limite_bajo) | (g[v] > limite_alto)).sum())
            filas.append({'dataset': ds, 'variable': v, 'outliers': n, 'documentos': len(g),
                          'porcentaje': round(100 * n / len(g), 1),
                          'limite_inferior': round(limite_bajo, 2), 'limite_superior': round(limite_alto, 2)})
    outliers = pd.DataFrame(filas)
    outliers.to_csv(Path(salida) / 'outliers_iqr.csv', index=False, encoding='utf-8-sig')
    print('  tablas: estadisticas_descriptivas.csv, outliers_iqr.csv')
    return describe, outliers

F = types.SimpleNamespace(AZUL=AZUL, NARANJA=NARANJA, AQUA=AQUA, GRIS=GRIS, FONDO=FONDO, TEXTO=TEXTO, TEXTO2=TEXTO2, REJILLA=REJILLA, COLOR=COLOR, NOMBRE=NOMBRE, CAMPOS=CAMPOS, es_num=es_num, guardar=guardar, leer_csv=leer_csv, fig_particiones=fig_particiones, fig_resolucion=fig_resolucion, fig_regiones=fig_regiones, fig_cobertura=fig_cobertura, fig_clases_lineas=fig_clases_lineas, fig_cer=fig_cer, tipo_error=tipo_error, fig_errores=fig_errores, fig_correlaciones=fig_correlaciones, PATRONES_FECHA=PATRONES_FECHA, patron=patron, fig_formatos_fecha=fig_formatos_fecha, tablas=tablas)

In [ ]:
import json, re, hashlib, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as Img, Markdown
warnings.filterwarnings('ignore', category=FutureWarning)
%matplotlib inline
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
import copy, time
from lector_facturas.common import FIELDS, write_json
from lector_facturas.ocr import transcribe
from lector_facturas import consensus, candidate_rules
from lector_facturas.evaluation import evaluate
from lector_facturas.experiments import bootstrap_delta, validation_samples
N_OPT, N_CONTROL = 30, 20   # por dataset; las mismas muestras que usa 04 (comparten la caché OCR)
CACHE = SALIDAS / 'cache_ocr'
RESULTADOS = SALIDAS / 'modelado'; RESULTADOS.mkdir(parents=True, exist_ok=True)
FIG = SALIDAS / 'figuras'; FIG.mkdir(parents=True, exist_ok=True)

# Etapas del extractor que se activan desde configs/definitivo.json
ETAPAS = ['printed_improvements', 'accounting_improvements', 'amount_context', 'spatial_amounts', 'semantic_amounts',
          'document_consistency', 'structured_fields', 'observed_accounting', 'confidence_rescue']

## 2. Muestra de validación
Se usan las muestras de optimización y control del notebook 04 (`experiments.validation_samples`): solo la partición `validation`, separadas por hash y sin imágenes que también estén en el test oficial. Aquí se juntan para tener más soporte por campo.

> Desde la versión «resumen contable», `configs/documentos_revisados.json` registra 1.461 documentos revisados durante el desarrollo, incluido casi todo el test oficial. Por eso estas cifras sirven para **comparar versiones**, no para afirmar desempeño en documentos nunca vistos.

In [ ]:
opt, control = validation_samples(FILAS, N_OPT, N_CONTROL, SEED)
muestra = opt + control
revisados = sum(r['id'] in REVISADOS or r['sha256'] in set(REVISADOS.values()) for r in muestra)
soporte = pd.DataFrame([{'dataset': r['dataset'], 'campo': f} for r in muestra for f in FIELDS if r['gold'][f]['status'] == 'annotated'])
print('Documentos:', Counter(r['dataset'] for r in muestra), '| ya revisados en el desarrollo:', revisados)
display(soporte.value_counts().rename('campos anotados').reset_index().pivot(index='campo', columns='dataset', values='campos anotados').fillna(0).astype(int))

## 3. Lectura OCR (una sola vez)
Se leen las 10 vistas de cada documento (más las de rescate cuando se activan) y se guardan en caché. La primera vez puede tardar 30–60 minutos con 100 documentos en CPU; reutiliza la caché del notebook 02 y la deja lista para 04. Los cuatro modelos usan **las mismas lecturas**, así que las diferencias se deben solo al extractor.

In [ ]:
lecturas, errores_ocr = {}, {}
inicio = time.perf_counter()
for i, fila in enumerate(muestra, 1):
    try:
        lecturas[fila['id']] = transcribe(fila['image'], CONFIG['ocr'], CACHE)
    except Exception as exc:
        errores_ocr[fila['id']] = str(exc)
    if i % 10 == 0 or i == len(muestra):
        print(f'{i}/{len(muestra)} · {time.perf_counter() - inicio:.0f} s')
print('Errores de OCR:', len(errores_ocr))

## 4. Modelos
Cada modelo es una función que recibe la lectura OCR y devuelve los seis campos. Todos se evalúan con `evaluation.evaluate`, la misma función del pipeline: exactitud, precisión, recall y F1 por campo; un valor equivocado cuenta como FP y FN; un campo sin anotación no cuenta.

In [ ]:
def es_vista_v7(vista):
    """Las 5 vistas del modelo anterior: pesos fast, solo inglés, sin rescate."""
    o = vista['options']
    return not vista.get('rescue') and o.get('language', 'eng') == 'eng' and Path(o.get('tessdata_dir', '')).name == 'fast'


def solo_vistas(ocr_resultado, condicion):
    """Copia de la lectura con un subconjunto de vistas (simula no haberlas leído)."""
    elegidas = [v for v in ocr_resultado['views'] if condicion(v)]
    principal = next((v for v in elegidas if consensus.is_primary(v)), elegidas[0])
    return dict(principal, views=elegidas)


def vista_principal(ocr_resultado):
    """Solo la lectura ampliada · PSM 6 · fast, como si el sistema leyera una vez."""
    principal = next(v for v in ocr_resultado['views'] if consensus.is_primary(v) and es_vista_v7(v))
    return dict(principal, views=[principal])


CONFIG_V7 = dict(CONFIG, **{e: False for e in ETAPAS})
CONFIG_SIN_RESUMEN = dict(CONFIG, observed_accounting=False, confidence_rescue=False)
MODELOS = {
    'M0 reglas iniciales (1 vista)': lambda o: candidate_rules.extract(vista_principal(o), CONFIG),
    'M1 V7 anterior (5 vistas fast)': lambda o: consensus.extract(solo_vistas(o, es_vista_v7), CONFIG_V7),
    'M2 actual sin resumen contable': lambda o: consensus.extract(o, CONFIG_SIN_RESUMEN),
    'M3 resumen contable (final)': lambda o: consensus.extract(o, CONFIG),
}

def evaluar(nombre, extractor, carpeta):
    predicciones = []
    for fila in muestra:
        o = lecturas.get(fila['id'])
        if o is None:
            predicciones.append({'id': fila['id'], 'dataset': fila['dataset'], 'fields': {}, 'status': 'error', 'text': ''})
            continue
        try:
            r = extractor(o)
            predicciones.append({'id': fila['id'], 'dataset': fila['dataset'], 'fields': r['fields'], 'alerts': r.get('alerts', []),
                                 'status': 'ok', 'text': o['text'], 'elapsed_seconds': None, 'cache_hit': True})
        except Exception as exc:
            predicciones.append({'id': fila['id'], 'dataset': fila['dataset'], 'fields': {}, 'status': 'error', 'error': str(exc), 'text': ''})
    resumen = evaluate(muestra, predicciones, CONFIG, Path(carpeta) / 'evaluation', calculate_cer=False)
    return resumen, predicciones

resultados = {}
for i, (nombre, extractor) in enumerate(MODELOS.items()):
    carpeta = RESULTADOS / f'M{i}'
    resumen, pred = evaluar(nombre, extractor, carpeta)
    resultados[nombre] = {'carpeta': carpeta, 'resumen': resumen, 'pred': pred}
    print(f"{nombre:34s} exactitud {resumen['micro_exact_match']:.1%}  ({resumen['evaluated_field_instances']} campos)")
nombres = list(resultados)
FINAL = nombres[-1]

## 5. Comparación de modelos

In [ ]:
filas = []
for nombre, r in resultados.items():
    m = pd.read_csv(r['carpeta'] / 'evaluation/metrics.csv', encoding='utf-8-sig')
    m = m[m.support > 0].assign(modelo=nombre)
    filas.append(m)
metricas = pd.concat(filas)
tabla = metricas.pivot_table(index=['dataset', 'field', 'support'], columns='modelo', values='exact_match').mul(100).round(1)
display(Markdown('**Exactitud por campo (%)**')); display(tabla)
tabla_f1 = metricas.pivot_table(index=['dataset', 'field'], columns='modelo', values='f1').round(3)
display(Markdown('**F1 por campo**')); display(tabla_f1)
global_ = pd.DataFrame({n: {'exactitud micro %': 100 * r['resumen']['micro_exact_match'],
                            'documentos procesados %': 100 * r['resumen']['execution_success_rate'],
                            'campos evaluados': r['resumen']['evaluated_field_instances']} for n, r in resultados.items()}).T.round(1)
display(Markdown('**Resumen global**')); display(global_)

In [ ]:
todo = metricas[metricas.dataset == 'ALL']
campos = [c for c in FIELDS if c in set(todo.field)]
fig, ax = plt.subplots(figsize=(9, 4))
ancho = .8 / len(resultados)
for i, (nombre, color) in enumerate(zip(resultados, [F.GRIS, F.NARANJA, F.AQUA, F.AZUL])):
    valores = [todo[(todo.modelo == nombre) & (todo.field == c)].exact_match.mean() * 100 for c in campos]
    ax.bar(np.arange(len(campos)) + i * ancho, valores, ancho, label=nombre, color=color)
ax.set_xticks(np.arange(len(campos)) + ancho * (len(resultados) - 1) / 2, campos); ax.set_ylabel('Exactitud (%)'); ax.set_ylim(0, 100)
ax.axhline(70, color='#52514e', ls='--', lw=1); ax.text(len(campos) - .5, 71, 'meta 70 %', fontsize=8, color='#52514e')
ax.set_title('Exactitud por campo y modelo (validación)', loc='left'); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG / 'comparacion_modelos.png', dpi=150); plt.show()

### 5.1 ¿La mejora es real o es ruido?
Bootstrap pareado por documento (`experiments.bootstrap_delta`): remuestrea documentos y calcula el intervalo del cambio de exactitud. Si el intervalo no incluye 0, la mejora es consistente en esta muestra.

In [ ]:
def carpeta_eval(r):
    return r['carpeta']
pares = [(nombres[0], FINAL), (nombres[1], FINAL), (nombres[2], FINAL)]
display(pd.DataFrame([dict(comparacion=f'{a} → {b}', **bootstrap_delta(carpeta_eval(resultados[a]), carpeta_eval(resultados[b])))
                      for a, b in pares]).round(2))

## 6. Ablación
Con las mismas lecturas se apaga **una cosa a la vez** frente al modelo final:
- cada **etapa** del extractor (interruptores de `configs/definitivo.json`);
- cada **grupo de vistas** OCR (pesos `best`, vista en español, vistas de rescate);
- cada **mecanismo del consenso** (`consensus.FLAGS`).

Un componente que no cambia nada en esta muestra no se elimina automáticamente: puede servir en otros formatos.

In [ ]:
base = resultados[FINAL]['resumen']['micro_exact_match']
variantes = {f'etapa: {e}': (lambda o, e=e: consensus.extract(o, dict(CONFIG, **{e: False}))) for e in ETAPAS}
variantes.update({
    'vistas: sin pesos best': lambda o: consensus.extract(solo_vistas(o, lambda v: Path(v['options'].get('tessdata_dir', '')).name != 'best'), CONFIG),
    'vistas: sin vista en español': lambda o: consensus.extract(solo_vistas(o, lambda v: '+spa' not in v['options'].get('language', '')), CONFIG),
    'vistas: sin rescate': lambda o: consensus.extract(solo_vistas(o, lambda v: not v.get('rescue')), CONFIG),
    'vistas: solo las 5 fast': lambda o: consensus.extract(solo_vistas(o, es_vista_v7), CONFIG),
})
ablacion = []
for nombre, extractor in variantes.items():
    resumen, _ = evaluar(nombre, extractor, RESULTADOS / ('ablacion_' + re.sub(r'\W+', '_', nombre)))
    ablacion.append({'componente apagado': nombre, 'exactitud %': 100 * resumen['micro_exact_match'],
                     'cambio (pp)': 100 * (resumen['micro_exact_match'] - base)})
originales = dict(consensus.FLAGS)
for flag in originales:
    consensus.FLAGS.update(originales); consensus.FLAGS[flag] = False
    try:
        resumen, _ = evaluar(flag, lambda o: consensus.extract(o, CONFIG), RESULTADOS / f'ablacion_flag_{flag}')
        ablacion.append({'componente apagado': f'consenso: {flag}', 'exactitud %': 100 * resumen['micro_exact_match'],
                         'cambio (pp)': 100 * (resumen['micro_exact_match'] - base)})
    finally:
        consensus.FLAGS.update(originales)
ablacion = pd.DataFrame(ablacion).sort_values('cambio (pp)').round(2)
ablacion.to_csv(RESULTADOS / 'ablacion.csv', index=False, encoding='utf-8-sig')
display(ablacion)
fig, ax = plt.subplots(figsize=(8, 0.35 * len(ablacion) + 1.2))
ax.barh(ablacion['componente apagado'], ablacion['cambio (pp)'], color=[F.NARANJA if v < 0 else F.AQUA for v in ablacion['cambio (pp)']])
ax.axvline(0, color='#52514e', lw=1); ax.set_xlabel('Cambio de exactitud al apagarlo (puntos porcentuales)')
ax.set_title('Ablación: aporte de cada componente', loc='left'); ax.invert_yaxis()
plt.tight_layout(); fig.savefig(FIG / 'ablacion.png', dpi=150); plt.show()

## 7. Análisis de errores del modelo final
Separamos los errores en **omisiones** (no devolvió valor) y **valores incorrectos**, como se hizo con la línea base de Semana 2.

In [ ]:
errores = pd.read_csv(resultados[FINAL]['carpeta'] / 'evaluation/errors.csv', encoding='utf-8-sig')
errores_m0 = pd.read_csv(resultados[nombres[0]]['carpeta'] / 'evaluation/errors.csv', encoding='utf-8-sig')
for nombre, e in [(nombres[0], errores_m0), (FINAL, errores)]:
    e = e.assign(tipo=F.tipo_error(e))
    display(Markdown(f'**{nombre}**')); display(e.tipo.value_counts().rename('campos').to_frame().T)
F.fig_errores(errores, FIG)
display(Img(filename=str(FIG / '07_errores_linea_base.png')))

In [ ]:
# Ejemplos de valores incorrectos (dataset público; sin datos privados).
incorrectos = errores[(errores.correct == 0) & errores.predicted.notna()]
display(incorrectos[['id', 'field', 'gold', 'predicted', 'gold_normalized', 'predicted_normalized']].head(12))
alertas = Counter(a.split(':')[-1] if ':' in a else a for p in resultados[FINAL]['pred'] for a in p.get('alerts', []))
display(pd.Series(alertas).sort_values(ascending=False).rename('apariciones').to_frame())

## 8. Tiempo de OCR por vista
Más vistas mejoran la exactitud a cambio de tiempo. Se mide con las lecturas que no venían de caché.

In [ ]:
def nombre_vista(vista):
    o = vista['options']
    modelo = 'esp' if '+spa' in o.get('language', 'eng') else Path(o.get('tessdata_dir', '')).name
    return f"{o['preprocess']}·psm{o['psm']}·{modelo}" + ('·rescate' if vista.get('rescue') else '')

tiempos = pd.DataFrame([{'id': doc, 'dataset': doc.split('/')[0], 'vista': nombre_vista(v), 'v7': es_vista_v7(v),
                         'segundos': v['ocr_seconds']}
                        for doc, o in lecturas.items() for v in o['views'] if not v.get('cache_hit')])
if tiempos.empty:
    print('Todas las lecturas vinieron de caché; borra la carpeta cache_ocr para medir tiempos.')
else:
    por_doc = tiempos.groupby(['dataset', 'id']).agg(todas_las_vistas=('segundos', 'sum'))
    por_doc['5_vistas_v7'] = tiempos[tiempos.v7].groupby(['dataset', 'id']).segundos.sum()
    por_doc['solo_principal'] = tiempos[tiempos.vista == 'resize·psm6·fast'].groupby(['dataset', 'id']).segundos.sum()
    display(por_doc.groupby('dataset').median().round(2).rename(columns=lambda c: c + ' (mediana, s)'))
    display(tiempos.groupby('vista').segundos.median().round(2).rename('segundos (mediana)').to_frame())
    print('Las vistas se leen con 2 procesos en paralelo (ocr.workers = 2): el tiempo real por documento es menor que la suma.')

## 9. Comparación con una técnica aprendida
En la Ficha técnica (Semana 2) se planteó comparar las reglas con un modelo que **aprende** a ubicar los campos (LayoutLMv3). Sin GPU disponible, se compara con su versión liviana, la misma idea del diagnóstico de la Semana 3: una **red neuronal (MLP) que clasifica cada línea del OCR** a partir de su texto y su posición ([`src/line_classifier.py`](../src/line_classifier.py)).

Para que la comparación sea justa:
- La red y la línea base **M0** usan **la misma lectura OCR** (una vista: ampliada · PSM 6 · `fast`). La diferencia entre ambas es solo el extractor: reglas escritas a mano frente a una red entrenada.
- Se entrena con la partición `train`, se eligen los hiperparámetros con `validation` (lecturas reales de Tesseract, con sus errores) y el test se usa **una sola vez** al final, con la misma muestra del notebook 05.
- Las etiquetas son **débiles**: una línea es positiva si contiene el valor anotado (los datasets anotan valores, no líneas).

Esta sección aplica la retroalimentación del diagnóstico de la Semana 3: se cambia **una cosa a la vez**, se elige con **validación y no con el test**, se justifican los **umbrales** del diagnóstico con una sensibilidad, se trazan **curvas de aprendizaje y de validación de la red**, se repite con **varias semillas** y se guardan los **historiales** en archivos.

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, str(CODIGO))
from src import line_classifier as LC
from lector_facturas.experiments import random_known_sample
SEMILLAS = [42, 7, 2024]                 # curvas y escalera
SEMILLAS_FINAL = [42, 7, 2024, 11, 123]  # evaluación final
CAMPO_DIAGNOSTICO = 'total'              # el campo que más importa contablemente y está anotado en ambos datasets
ML = SALIDAS / 'comparacion_ml'; ML.mkdir(parents=True, exist_ok=True)
CACHE_1 = SALIDAS / 'cache_ocr_una_vista'
NORMALIZACION = CONFIG['normalization']
VISTA = next(v for v in CONFIG['ocr']['ensemble_views']
             if v.get('preprocess') == 'resize' and v.get('psm') == 6 and Path(v.get('tessdata_dir', '')).name == 'fast')
OCR_1 = dict(CONFIG['ocr'], ensemble_views=[VISTA], adaptive_rescue=dict(CONFIG['ocr']['adaptive_rescue'], enabled=False))

prohibidas = {r['sha256'] for r in FILAS if r['split'] in ('test', 'quarantine')}
entrenamiento = [r for r in FILAS if r['split'] == 'train' and r['sha256'] not in prohibidas]
validacion = [r for r in FILAS if r['split'] == 'validation' and r['sha256'] not in prohibidas]
test_ml, plan_test_ml = random_known_sample(FILAS, 50, seed=SEED)   # misma muestra que el notebook 05


def leer_una_vista(filas):
    def uno(fila):
        try:
            return fila['id'], transcribe(fila['image'], OCR_1, CACHE_1)
        except Exception as exc:
            print('Sin OCR:', fila['id'], exc)
            return fila['id'], None
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        return {i: o for i, o in pool.map(uno, filas) if o is not None}


inicio = time.perf_counter()
L_tr, L_va, L_te = leer_una_vista(entrenamiento), leer_una_vista(validacion), leer_una_vista(test_ml)
print(f'OCR de una vista: {len(L_tr)} train, {len(L_va)} validación, {len(L_te)} test ({(time.perf_counter() - inicio) / 60:.1f} min)')
D_tr, D_va, D_te = (LC.preparar_documentos(f, l, NORMALIZACION) for f, l in [(entrenamiento, L_tr), (validacion, L_va), (test_ml, L_te)])
CZ = LC.Caracterizador().ajustar([l['texto'] for d in D_tr for l in d['lineas']])
cobertura = []
for base in ('sroie', 'wildreceipt'):
    for campo in LC.CAMPOS_APRENDIDOS:
        etiq = [d['etiquetas'][campo] for d in D_tr if d['dataset'] == base and d['etiquetas'][campo] is not None]
        if etiq:
            cobertura.append({'dataset': base, 'campo': campo, 'documentos anotados': len(etiq),
                              '% con una línea etiquetada': round(100 * np.mean([e.any() for e in etiq]), 1),
                              'líneas positivas por documento': round(float(np.mean([e.sum() for e in etiq])), 2)})
cobertura = pd.DataFrame(cobertura)
cobertura.to_csv(ML / 'cobertura_etiquetas_debiles.csv', index=False, encoding='utf-8-sig')
display(cobertura)
print('Líneas por documento:', round(np.mean([len(d['lineas']) for d in D_tr]), 1), '| rasgos por línea:', CZ.transformar(D_tr[0]['lineas'], 'total').shape[1])

Si un valor anotado no aparece en ninguna línea (el OCR lo leyó mal), el documento se conserva con todas sus líneas negativas: así la red también aprende cuándo no hay un buen candidato. Ese porcentaje es además el **techo** de esta técnica con una sola lectura: no puede extraer un valor que el OCR no leyó.

### 9.1 Diagnóstico de overfitting: un cambio a la vez
Cada modelo agrega **un solo cambio** al anterior (en la Semana 3, E1b cambió a la vez el tamaño de la red y las épocas). Se entrena la red del campo `total` con tres semillas y se registran, época por época, la pérdida y el F1 de entrenamiento y validación.

| Modelo | Cambio respecto al anterior |
|---|---|
| R0 | Red grande (512 → 256), sin regularización, 40 épocas |
| R1 | + regularización L2 (α = 0,001) |
| R2 | + red pequeña (128) |
| R3 | + early stopping (paciencia 5, conserva la mejor época) |

In [ ]:
ESCALERA = {
    'R0 red grande sin regularización': LC.ConfigRed(capas=(512, 256), l2=1e-6, epocas=40, paciencia=None),
    'R1 + L2': LC.ConfigRed(capas=(512, 256), l2=1e-3, epocas=40, paciencia=None),
    'R2 + red pequeña': LC.ConfigRed(capas=(128,), l2=1e-3, epocas=40, paciencia=None),
    'R3 + early stopping': LC.ConfigRed(capas=(128,), l2=1e-3, epocas=40, paciencia=5),
}
A_diag = LC.armar_conjunto(D_tr, CAMPO_DIAGNOSTICO, CZ)
B_diag = LC.armar_conjunto(D_va, CAMPO_DIAGNOSTICO, CZ)
print('Líneas train:', A_diag.X.shape, '| validación:', B_diag.X.shape, '| positivas train:', round(100 * A_diag.y.mean(), 2), '%')
historiales, resumenes = [], {}
inicio = time.perf_counter()
for nombre, cfg in ESCALERA.items():
    for semilla in SEMILLAS:
        _, h = LC.entrenar(A_diag, B_diag, cfg, semilla, nombre)
        tabla = h.tabla()
        historiales.append(tabla.assign(modelo=nombre, semilla=semilla))
        resumenes[(nombre, semilla)] = LC.cuantificar(tabla, restaurada=bool(cfg.paciencia))
    print(f'{nombre}: {(time.perf_counter() - inicio) / 60:.1f} min', flush=True)
historial_escalera = pd.concat(historiales, ignore_index=True)
historial_escalera.to_csv(ML / 'historial_escalera.csv', index=False, encoding='utf-8-sig')
diag = pd.DataFrame([{'modelo': n, 'semilla': s, **r, 'diagnostico': LC.diagnosticar(r)} for (n, s), r in resumenes.items()])
diag.to_csv(ML / 'diagnostico_escalera.csv', index=False, encoding='utf-8-sig')
resumen_escalera = diag.groupby('modelo', sort=False).agg(
    mejor_epoca=('mejor_epoca', 'mean'), subida_perdida_val_pct=('subida_perdida_val_pct', 'mean'),
    f1_train=('f1_train_final', 'mean'), f1_val=('f1_val_final', 'mean'), f1_val_std=('f1_val_final', 'std'),
    brecha_f1=('brecha_f1', 'mean'), diagnostico=('diagnostico', lambda x: x.value_counts().idxmax())).round(3)
resumen_escalera.to_csv(ML / 'resumen_escalera.csv', encoding='utf-8-sig')
display(resumen_escalera)

In [ ]:
fig, ejes = plt.subplots(2, 4, figsize=(15, 6.2), sharex=True)
for j, nombre in enumerate(ESCALERA):
    h = historial_escalera[(historial_escalera.modelo == nombre) & (historial_escalera.semilla == SEMILLAS[0])]
    ax = ejes[0, j]
    ax.plot(h.epoca, h.perdida_train, color=AZUL, ls='--', label='train')
    ax.plot(h.epoca, h.perdida_val, color=NARANJA, label='validación')
    ax.set_title(nombre, fontsize=10); ax.set_ylabel('pérdida (log-loss)' if j == 0 else '')
    mejor = h.loc[h.perdida_val.idxmin(), 'epoca']; ax.axvline(mejor, color=GRIS, ls=':', lw=1)
    ax = ejes[1, j]
    ax.plot(h.epoca, h.f1_train, color=AZUL, ls='--', label='train')
    ax.plot(h.epoca, h.f1_val, color=NARANJA, label='validación')
    ax.set_ylim(0, 1.02); ax.set_xlabel('época'); ax.set_ylabel('F1 (línea con el total)' if j == 0 else '')
ejes[0, 0].legend()
fig.suptitle(f'Curvas por época de la red del campo «{CAMPO_DIAGNOSTICO}» (semilla {SEMILLAS[0]}; la línea punteada marca la mejor época)', fontsize=11)
guardar(fig, FIG, 'ml_curvas_escalera.png'); display(Img(filename=str(FIG / 'ml_curvas_escalera.png')))

### 9.2 ¿Dependen las conclusiones de los umbrales?
El diagnóstico usa dos umbrales: overfitting si la pérdida de validación sube más de **10 %** sobre su mínimo o si la brecha de F1 entre entrenamiento y validación supera **0,10**. En lugar de defenderlos solo con palabras, se repite el diagnóstico con umbrales más estrictos y más laxos: si el veredicto de cada modelo no cambia en la mayoría de combinaciones, la conclusión no depende de la elección exacta.

In [ ]:
medios = {n: g.drop(columns=['modelo', 'semilla', 'diagnostico', 'inicio_sobreajuste']).mean().to_dict() for n, g in diag.groupby('modelo', sort=False)}
sens = LC.sensibilidad_umbrales(medios, subidas=(5, 10, 15, 20, 30), brechas=(0.05, 0.10, 0.15, 0.20))
sens.to_csv(ML / 'sensibilidad_umbrales.csv', index=False, encoding='utf-8-sig')
display(sens.pivot_table(index='modelo', columns='diagnostico', values='subida_max_%', aggfunc='count', fill_value=0)
        .reindex(list(ESCALERA)).rename_axis(columns='combinaciones de umbrales (de 20) con cada veredicto'))

### 9.3 Curva de aprendizaje y curvas de validación de la red
- **Aprendizaje:** F1 de entrenamiento y validación con 10 %, 25 %, 50 % y 100 % de los documentos de entrenamiento (configuración R3). Si la validación sigue subiendo al final, más datos ayudarían.
- **Validación:** F1 al variar el tamaño de la capa oculta y la regularización L2, uno a la vez y con el resto en R3.

In [ ]:
base = ESCALERA['R3 + early stopping']
filas_aprendizaje = []
for fraccion in (0.10, 0.25, 0.50, 1.0):
    for semilla in SEMILLAS:
        rng = np.random.default_rng(semilla)
        sub = [D_tr[i] for i in sorted(rng.choice(len(D_tr), max(5, int(fraccion * len(D_tr))), replace=False))]
        _, h = LC.entrenar(LC.armar_conjunto(sub, CAMPO_DIAGNOSTICO, CZ), B_diag, base, semilla)
        r = LC.cuantificar(h.tabla(), restaurada=True)
        filas_aprendizaje.append({'fraccion': fraccion, 'documentos': len(sub), 'semilla': semilla,
                                  'f1_train': r['f1_train_final'], 'f1_val': r['f1_val_final']})
curva_aprendizaje = pd.DataFrame(filas_aprendizaje)
curva_aprendizaje.to_csv(ML / 'curva_aprendizaje.csv', index=False, encoding='utf-8-sig')

filas_validacion = []
barridos = {'capas': [(16,), (64,), (128,), (256,), (512,)], 'l2': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]}
for parametro, valores in barridos.items():
    for valor in valores:
        cfg = LC.ConfigRed(**{**base.__dict__, parametro: valor})
        for semilla in SEMILLAS:
            _, h = LC.entrenar(A_diag, B_diag, cfg, semilla)
            r = LC.cuantificar(h.tabla(), restaurada=True)
            filas_validacion.append({'parametro': parametro, 'valor': str(valor[0] if parametro == 'capas' else valor),
                                     'semilla': semilla, 'f1_train': r['f1_train_final'], 'f1_val': r['f1_val_final']})
curva_validacion = pd.DataFrame(filas_validacion)
curva_validacion.to_csv(ML / 'curvas_validacion.csv', index=False, encoding='utf-8-sig')

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (titulo, datos, x) in zip(ejes, [('Curva de aprendizaje', curva_aprendizaje, 'documentos'),
                                          ('Tamaño de la capa oculta', curva_validacion[curva_validacion.parametro == 'capas'], 'valor'),
                                          ('Regularización L2 (α)', curva_validacion[curva_validacion.parametro == 'l2'], 'valor')]):
    g = datos.groupby(x, sort=False)[['f1_train', 'f1_val']].agg(['mean', 'std'])
    etiquetas = [str(v) for v in g.index]
    for col, color, nombre in [('f1_train', AZUL, 'train'), ('f1_val', NARANJA, 'validación')]:
        ax.errorbar(range(len(g)), g[(col, 'mean')], yerr=g[(col, 'std')], color=color, marker='o', capsize=3, label=nombre)
    ax.set_xticks(range(len(g)), etiquetas); ax.set_title(titulo); ax.set_ylim(0, 1.02); ax.set_ylabel('F1')
ejes[0].set_xlabel('documentos de entrenamiento'); ejes[0].legend()
fig.suptitle('Media ± desviación de 3 semillas', fontsize=10)
guardar(fig, FIG, 'ml_curvas_aprendizaje_validacion.png'); display(Img(filename=str(FIG / 'ml_curvas_aprendizaje_validacion.png')))
display(curva_aprendizaje.groupby('documentos')[['f1_train', 'f1_val']].agg(['mean', 'std']).round(3))
display(curva_validacion.groupby(['parametro', 'valor'], sort=False)[['f1_train', 'f1_val']].agg(['mean', 'std']).round(3))

### 9.4 Selección con validación (el test no se mira)
Se entrena una red por campo y se mide la exactitud **de extremo a extremo** en `validation` con la misma función del pipeline (`evaluation.evaluate`). Se comparan R3 y la mejor capa y regularización de las curvas de validación, y el umbral de probabilidad por debajo del cual el campo queda vacío.

In [ ]:
def entrenar_campos(cfg, semilla, documentos=D_tr):
    redes = {}
    for campo in LC.CAMPOS_APRENDIDOS:
        redes[campo], _ = LC.entrenar(LC.armar_conjunto(documentos, campo, CZ), LC.armar_conjunto(D_va, campo, CZ), cfg, semilla)
    return redes


def evaluar_ml(redes, umbral, documentos, carpeta):
    clasificador = LC.ClasificadorLineas(CZ, redes, umbral)
    return evaluate([d['fila'] for d in documentos], clasificador.predecir(documentos), CONFIG, Path(carpeta) / 'evaluation', calculate_cer=False)


medias = curva_validacion.groupby(['parametro', 'valor']).f1_val.mean()
mejor_capas = int(medias['capas'].idxmax())
mejor_l2 = float(medias['l2'].idxmax())
candidatas = {'R3': base, f'capa {mejor_capas}': LC.ConfigRed(**{**base.__dict__, 'capas': (mejor_capas,)}),
              f'L2 {mejor_l2:g}': LC.ConfigRed(**{**base.__dict__, 'l2': mejor_l2})}
filas_sel = []
for nombre, cfg in candidatas.items():
    redes = entrenar_campos(cfg, SEMILLAS[0])
    for umbral in (0.2, 0.35, 0.5, 0.65):
        s = evaluar_ml(redes, umbral, D_va, ML / 'seleccion' / f'{nombre}_{umbral}')
        filas_sel.append({'configuracion': nombre, 'detalle': cfg.etiqueta(), 'umbral': umbral,
                          'exactitud_validacion': 100 * s['micro_exact_match'], 'campos': s['evaluated_field_instances']})
seleccion = pd.DataFrame(filas_sel).round(2)
seleccion.to_csv(ML / 'seleccion_validacion.csv', index=False, encoding='utf-8-sig')
elegida = seleccion.sort_values(['exactitud_validacion', 'umbral'], ascending=[False, False]).iloc[0]
CFG_ML, UMBRAL_ML = candidatas[elegida.configuracion], float(elegida.umbral)
display(seleccion.pivot(index='configuracion', columns='umbral', values='exactitud_validacion'))
print('Elegida con validación:', elegida.configuracion, '|', CFG_ML.etiqueta(), '| umbral', UMBRAL_ML)

### 9.5 Evaluación final en el test (una sola vez)
La red elegida se entrena con **cinco semillas** y se evalúa en la muestra del test del notebook 05. Se compara con **M0** (reglas, misma lectura de una vista) y con el **modelo final M3** (10 vistas, cifra del notebook 05). Diferencias de 1 a 2 puntos entre semillas no se interpretan como reales: se reporta la media con su intervalo.

In [ ]:
from scipy import stats
filas_final, carpetas_ml = [], {}
for semilla in SEMILLAS_FINAL:
    redes = entrenar_campos(CFG_ML, semilla)
    carpeta = ML / 'test' / f'semilla_{semilla}'
    s = evaluar_ml(redes, UMBRAL_ML, D_te, carpeta)
    carpetas_ml[semilla] = carpeta
    filas_final.append({'semilla': semilla, 'exactitud': 100 * s['micro_exact_match'], 'procesados': 100 * s['execution_success_rate']})
por_semilla = pd.DataFrame(filas_final)
por_semilla.to_csv(ML / 'test_por_semilla.csv', index=False, encoding='utf-8-sig')
media, desv = por_semilla.exactitud.mean(), por_semilla.exactitud.std(ddof=1)
margen = stats.t.ppf(0.975, len(por_semilla) - 1) * desv / np.sqrt(len(por_semilla))

predicciones_m0 = []
for d in D_te:
    try:
        r = candidate_rules.extract(vista_principal(L_te[d['id']]), CONFIG)
        predicciones_m0.append({'id': d['id'], 'dataset': d['dataset'], 'fields': r['fields'], 'status': 'ok', 'text': '', 'cache_hit': True})
    except Exception as exc:
        predicciones_m0.append({'id': d['id'], 'dataset': d['dataset'], 'fields': {}, 'status': 'error', 'error': str(exc), 'text': ''})
s_m0 = evaluate([d['fila'] for d in D_te], predicciones_m0, CONFIG, ML / 'test' / 'M0' / 'evaluation', calculate_cer=False)
m3 = pd.read_csv(CODIGO / 'results/metrics/evaluacion/comparacion_modelos.csv', encoding='utf-8-sig', index_col=0)
fila_m3 = m3.loc[[i for i in m3.index if i.startswith('M3')][0]]
final_ml = pd.DataFrame([
    {'modelo': 'M0 reglas iniciales (1 vista)', 'lectura OCR': '1 vista', 'exactitud': 100 * s_m0['micro_exact_match'], 'ic95': '—'},
    {'modelo': f'Red MLP sobre líneas ({CFG_ML.etiqueta()})', 'lectura OCR': '1 vista', 'exactitud': media,
     'ic95': f'{media - margen:.1f} – {media + margen:.1f} (5 semillas)'},
    {'modelo': 'M3 resumen contable (final)', 'lectura OCR': '10 vistas', 'exactitud': float(fila_m3['exactitud %']), 'ic95': '75,0 – 86,9 (bootstrap, notebook 05)'},
]).round(1)
final_ml.to_csv(ML / 'comparacion_tecnicas_test.csv', index=False, encoding='utf-8-sig')
display(final_ml)
delta = bootstrap_delta(ML / 'test' / 'M0', carpetas_ml[SEMILLAS_FINAL[0]])
print('Red − M0 (misma lectura, semilla', SEMILLAS_FINAL[0], '):', {k: round(v, 2) if isinstance(v, float) else v for k, v in delta.items()})
por_campo = []
for nombre, carpeta in [('M0 reglas (1 vista)', ML / 'test' / 'M0'), ('Red MLP (1 vista)', carpetas_ml[SEMILLAS_FINAL[0]])]:
    m = pd.read_csv(Path(carpeta) / 'evaluation/metrics.csv', encoding='utf-8-sig').query("support > 0 and dataset == 'ALL'")
    por_campo += [{'modelo': nombre, 'campo': r.field, 'exactitud': round(100 * r.exact_match, 1), 'f1': round(r.f1, 3)} for r in m.itertuples()]
por_campo = pd.DataFrame(por_campo)
por_campo.to_csv(ML / 'comparacion_tecnicas_por_campo.csv', index=False, encoding='utf-8-sig')
display(por_campo.pivot(index='campo', columns='modelo', values='exactitud'))
write_json(ML / 'comparacion_tecnicas.json', {'configuracion_red': CFG_ML.__dict__ | {'capas': list(CFG_ML.capas)}, 'umbral': UMBRAL_ML,
           'semillas': SEMILLAS_FINAL, 'media': media, 'ic95': [media - margen, media + margen], 'm0': 100 * s_m0['micro_exact_match'],
           'delta_red_menos_m0': delta, 'documentos_test': len(D_te), 'documentos_train': len(D_tr), 'documentos_validacion': len(D_va)})

### 9.6 Por qué el modelo final no es la red
_Se completa con los resultados de la ejecución._

## 10. Modelo final
El modelo elegido es **M3: resumen contable**. No hay un archivo `.pkl`: el "modelo" queda definido por

- los pesos de Tesseract `eng.traineddata` (`tessdata_fast` y `tessdata_best`) y `spa.traineddata`, descargados de revisiones fijas y verificados por SHA-256 (`frozen_runtime.bind_models`);
- la configuración `configs/definitivo.json` (vistas, etapas activas, normalización y validaciones); y
- el código de `src/lector_facturas/` (versión registrada por commit y por hash de cada archivo).

La celda siguiente guarda esa definición con sus hashes, para que cualquier resultado posterior se pueda atribuir a esta versión exacta.

In [ ]:
pesos = sorted({(Path(v['tessdata_dir']).name, v.get('language', 'eng'), v['expected_model_sha256'])
                for v in CONFIG['ocr']['ensemble_views'] + CONFIG['ocr']['rescue_views']})
codigo = {str(p.relative_to(RAIZ)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((RAIZ / 'src/lector_facturas').glob('*.py'))}
definicion = {'modelo': 'Resumen contable: consenso de 10 vistas Tesseract + rescate adaptativo', 'revision_codigo': REVISION,
              'pesos_ocr': [{'pesos': p, 'idioma': l, 'sha256_eng': h} for p, l, h in pesos],
              'configuracion': RAW_CONFIG, 'codigo_sha256': codigo, 'flags': dict(consensus.FLAGS),
              'validacion': {n: {k: r['resumen'][k] for k in ('documents', 'micro_exact_match', 'execution_success_rate', 'evaluated_field_instances')}
                             for n, r in resultados.items()}}
(RESULTADOS / 'modelo_final.json').write_text(json.dumps(definicion, ensure_ascii=False, indent=2), encoding='utf-8')
print('Guardado en', RESULTADOS / 'modelo_final.json')
display(global_)

### Conclusiones
- La comparación usa las mismas lecturas OCR para los cuatro modelos, así que las diferencias se deben al extractor (y, en M1, a usar menos vistas).
- La tabla de la sección 5 y el bootstrap indican si el modelo final mejora a las versiones anteriores de forma consistente en esta muestra; la ablación muestra qué etapas, vistas y mecanismos explican esa mejora.
- Estas cifras son de **validación** y sirven para elegir el modelo. La evaluación final está en `05_evaluacion.ipynb`.
- `tax_id` y `document_number` no se pueden medir con estos datasets; necesitan la muestra propia.